# Rhino → MIDAS Pipeline Verification

This notebook exercises the **Rhino → MIDAS Civil NX** transfer end to end and
checks that every object type makes it across intact.

The pipeline is:

```
tagged .3dm  ──read_girder_model──▶  StructuralModel (the hub)
                                          │
                            build_deck ── adds deck / parapet dead loads
                                          │
                                   midas_payloads
                                          │
        ┌─────────────┬──────────┬────────┴────────┬──────────┬──────────┐
       UNIT          MATL       SECT              NODE       ELEM       CONS / STLD+CNLD
     (units)     (materials)  (sections)         (nodes)  (elements)   (supports / loads)
                                          │
                                   push_midas ──▶  live Civil NX session
```

What we verify: the **nodes, elements, sections, materials, supports, and
loads** that leave the Rhino model arrive in the MIDAS payloads with matching
counts, geometry, and assignments. The geometry of the deck and parapets is a
*display* product; their **dead loads** are the real contribution that must
transfer.

> The section/material/node/element/support payloads are **pure** — no live
> session needed, so the checks below run offline. The final live-push cell is
> guarded behind `RUN_MIDAS` and only fires against a running Civil NX.

## 1 · Read the tagged Rhino model

In [1]:
import warnings
warnings.simplefilter("ignore")  # deck params are optional on this file

from pathlib import Path
from civilpy.structural.rhino_gdr import read_girder_model

# The consultant-authored 4-span continuous bridge (REF-DESIGN): 5 girder
# lines, W24 sections that step up over the piers, curves broken at every
# shape transition.
RHINO_FILE = Path("../Notebooks/res/Rhino 3dms/REF-DESIGN.3dm")

bridge = read_girder_model(str(RHINO_FILE))
model = bridge.model
print(f"nodes      : {len(model.nodes)}")
print(f"elements   : {len(model.elements)}")
print(f"restraints : {len(model.restraints)}")
print(f"girder lines: {sorted(bridge.girder_lines)}")

nodes      : 55
elements   : 50
restraints : 25
girder lines: ['1', '2', '3', '4', '5']


### The interior bearings survive the read

Every girder line carries **5 bearings** (two abutments + three piers), so a
correct read gives `5 lines × 5 = 25` restraints. This is the regression guard
for the mid-element-bearing fix: the girder curves break at *shape transitions*,
not at the piers, so a naive reader attaches bearings only to curve endpoints
and silently drops every interior pier — collapsing the 4-span continuous unit
into five simple spans. 25 restraints means the piers made it.

In [2]:
import collections

by_line = collections.defaultdict(list)
for nid, r in model.restraints.items():
    x = model.nodes[r.node_id].x if hasattr(r, "node_id") else model.nodes[nid].x
    by_line[model.nodes[getattr(r, "node_id", nid)].y].append(round(x, 2))

assert len(model.restraints) == 25, "interior bearings were dropped!"
for y in sorted(by_line):
    print(f"girder @ y={y:5.1f} ft : bearings at x = {sorted(by_line[y])} ft")
print("\nPASS — 25 restraints, 5 per line (4-span continuous preserved)")

girder @ y=  0.0 ft : bearings at x = [0.21, 37.75, 100.25, 162.75, 200.71] ft
girder @ y=  7.0 ft : bearings at x = [0.21, 37.75, 100.25, 162.75, 200.71] ft
girder @ y= 14.0 ft : bearings at x = [0.0, 37.75, 100.25, 162.75, 200.5] ft
girder @ y= 21.0 ft : bearings at x = [0.0, 37.75, 100.25, 162.75, 200.5] ft
girder @ y= 28.0 ft : bearings at x = [0.0, 37.75, 100.25, 162.75, 200.5] ft

PASS — 25 restraints, 5 per line (4-span continuous preserved)


## 2 · Deck & parapets → dead loads

In [3]:
import tempfile, os
from civilpy.structural.rhino_deck import build_deck

deck_out = os.path.join(tempfile.mkdtemp(), "deck.3dm")
deck = build_deck(str(RHINO_FILE), out_path=deck_out,
                  deck_t_in=8.5, overhang_ft=3.5, parapet="BR-1 (36 in)")
print(f"deck slab      : {deck.width_ft:.1f} ft wide × {deck.length_ft:.1f} ft long, "
      f"{deck.deck_t_in:.1f} in thick")
print(f"girder spacing : {deck.girder_spacing_ft:.1f} ft  ({deck.n_girder_lines} lines)")
print(f"deck DC1       : {deck.deck_dc1_klf_interior:.3f} klf on an interior girder")
print(f"parapet        : {deck.parapet}  →  DC2 {deck.parapet_dc2_klf_each:.3f} klf each edge")
print(f"objects written: {deck.n_deck} slab, {deck.n_parapet} parapet(s), {deck.n_railing} rail(s)")

deck slab      : 35.0 ft wide × 200.7 ft long, 8.5 in thick
girder spacing : 7.0 ft  (5 lines)
deck DC1       : 0.744 klf on an interior girder
parapet        : BR-1 (36 in)  →  DC2 0.441 klf each edge
objects written: 1 slab, 2 parapet(s), 0 rail(s)


Lump the deck self-weight onto the girder nodes as a **DC** load case, using
each node's tributary length (half of every element that frames into it). This
is the deck's contribution entering the analysis model — the loads that must
show up in the MIDAS `STLD`/`CNLD` tables.

In [4]:
# tributary length per node from the connected elements
trib = collections.defaultdict(float)
for e in model.elements.values():
    a, b = model.nodes[e.node_a], model.nodes[e.node_b]
    L = ((a.x - b.x) ** 2 + (a.y - b.y) ** 2 + (a.z - b.z) ** 2) ** 0.5
    trib[e.node_a] += L / 2.0
    trib[e.node_b] += L / 2.0

w = deck.deck_dc1_klf_interior  # klf (interior tributary; exterior also carry DC2)
n_loads = 0
for nid, t in trib.items():
    model.add_load(nid, fz=-w * t, case="DC")
    n_loads += 1
print(f"applied {n_loads} nodal DC loads  (total ≈ {w * sum(trib.values()):.1f} kip)")

applied 55 nodal DC loads  (total ≈ 745.6 kip)


## 3 · Serialize to MIDAS payloads

In [5]:
from civilpy.structural.midas_models import (
    midas_payloads, hub_section_material_blocks,
)

payloads = midas_payloads(model)
print("tables in send order:", list(payloads))
for t in ("NODE", "ELEM", "CONS", "STLD", "CNLD"):
    print(f"  {t:5s}: {len(payloads.get(t, {}))} entr{'y' if len(payloads.get(t, {}))==1 else 'ies'}")

tables in send order: ['UNIT', 'MATL', 'SECT', 'NODE', 'ELEM', 'CONS', 'STLD', 'CNLD']
  NODE : 55 entries
  ELEM : 50 entries
  CONS : 25 entries
  STLD : 1 entry
  CNLD : 55 entries


In [6]:
# real per-shape SECT and per-grade MATL (replace the single placeholder)
blocks = hub_section_material_blocks(model)
print("distinct sections:", blocks["sect_by_shape"])
print("distinct grades  :", blocks["matl_by_grade"])
for shape, sid in blocks["sect_by_shape"].items():
    h, b, tw, tf, *_ = blocks["SECT"][str(sid)]["SECT_BEFORE"]["SECT_I"]["vSIZE"]
    print(f"  SECT {sid}: {shape:8s}  H={h} B={b} tw={tw} tf={tf}  (in)")

distinct sections: {'W24X104': 1, 'W24X131': 2, 'W24X162': 3}
distinct grades  : {'Grade 50': 1}
  SECT 1: W24X104   H=24.1 B=12.8 tw=0.5 tf=0.75  (in)
  SECT 2: W24X131   H=24.5 B=12.9 tw=0.605 tf=0.96  (in)
  SECT 3: W24X162   H=25.0 B=13.0 tw=0.705 tf=1.22  (in)


## 4 · Verify every object type transfers

The checks below confirm the payloads are a faithful, complete image of the
hub — matching counts, one-to-one id maps, correct geometry and assignments.

In [7]:
def check(name, ok, detail=""):
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}"
          + (f" — {detail}" if detail else ""))
    assert ok, name

report = {}

# NODE — count, 1-based ids, coordinates
nodes = payloads["NODE"]
check("NODE count == hub nodes", len(nodes) == len(model.nodes),
      f"{len(nodes)}")
check("NODE ids are 1..N", set(nodes) == {str(i) for i in range(1, len(nodes) + 1)})
hub_xyz = {(round(n.x, 4), round(n.y, 4), round(n.z, 4)) for n in model.nodes.values()}
pay_xyz = {(round(v["X"], 4), round(v["Y"], 4), round(v["Z"], 4)) for v in nodes.values()}
check("NODE coordinates round-trip", hub_xyz == pay_xyz)
report["NODE"] = len(nodes)

# ELEM — count, node refs valid, BEAM type
elems = payloads["ELEM"]
check("ELEM count == hub elements", len(elems) == len(model.elements))
ok_refs = all(1 <= e["NODE"][0] <= len(nodes) and 1 <= e["NODE"][1] <= len(nodes)
              for e in elems.values())
check("ELEM node refs in range", ok_refs)
check("ELEM all BEAM", all(e["TYPE"] == "BEAM" for e in elems.values()))
report["ELEM"] = len(elems)

# SECT / MATL — real, every element assigned
check("distinct SECT per shape", set(blocks["sect_by_shape"]) ==
      {e.section for e in model.elements.values() if e.section})
check("every element has a real SECT",
      all(sid is not None for sid, _ in blocks["elem_assign"].values()))
report["SECT"] = len(blocks["SECT"])
report["MATL"] = len(blocks["MATL"])

# CONS — one per restrained node, 7-char flag string
cons = payloads["CONS"]
check("CONS count == restraints", len(cons) == len(model.restraints))
ok_flag = all(len(v["ITEMS"][0]["CONSTRAINT"]) == 7 for v in cons.values())
check("CONS strings are 7-char", ok_flag)
report["CONS"] = len(cons)

# STLD / CNLD — the DC load case transferred
check("STLD has the DC case", any(c["NAME"] == "DC" for c in payloads["STLD"].values()))
n_cnld = sum(len(v["ITEMS"]) for v in payloads["CNLD"].values())
check("CNLD nodal loads present", n_cnld > 0, f"{n_cnld} loads")
report["loads"] = n_cnld

print("\nAll object types transferred:", report)

  [PASS] NODE count == hub nodes — 55
  [PASS] NODE ids are 1..N
  [PASS] NODE coordinates round-trip
  [PASS] ELEM count == hub elements
  [PASS] ELEM node refs in range
  [PASS] ELEM all BEAM
  [PASS] distinct SECT per shape
  [PASS] every element has a real SECT
  [PASS] CONS count == restraints
  [PASS] CONS strings are 7-char
  [PASS] STLD has the DC case
  [PASS] CNLD nodal loads present — 55 loads

All object types transferred: {'NODE': 55, 'ELEM': 50, 'SECT': 3, 'MATL': 1, 'CONS': 25, 'loads': 55}


### Decode the support fixities

MIDAS packs each support into a 7-flag string `Dx Dy Dz Rx Ry Rz Rw` (1 = fixed).
A **fixed** bearing holds longitudinal translation; an **expansion** bearing
frees it. The reader mapped `gdr.fixity` to exactly these — here is the proof
the distinction survives to MIDAS.

In [8]:
# integer node id -> (x, fixity string)
node_int = {n.id: i for i, n in enumerate(model.nodes.values(), start=1)}
int_x = {i: model.nodes[nid].x for nid, i in node_int.items()}

seen = {}
for k, v in cons.items():
    s = v["ITEMS"][0]["CONSTRAINT"]
    kind = "FIXED (Dx held)" if s[0] == "1" else "expansion (Dx free)"
    seen.setdefault((s, kind), []).append(round(int_x[int(k)], 1))

for (s, kind), xs in sorted(seen.items()):
    print(f"  {s}  {kind:22s}  at x = {sorted(set(xs))} ft  ×{len(xs)} bearings")

  0110000  expansion (Dx free)     at x = [0.0, 0.2, 37.8, 162.8, 200.5, 200.7] ft  ×20 bearings
  1110000  FIXED (Dx held)         at x = [100.2] ft  ×5 bearings


## 5 · Live push to Civil NX  *(guarded)*

Flip `RUN_MIDAS = True` with Civil NX open and the API enabled to push the
model and read it back. `push_midas` sends every table in order and returns a
per-table report; the read-back confirms the counts on the MIDAS side.

Leave it `False` for a dry run — it prints the payload sizes that *would* be
sent.

In [9]:
RUN_MIDAS = False

if RUN_MIDAS:
    from civilpy.structural.midas_models import push_midas
    from civilpy.structural import midas as mc

    report_live = push_midas(model)
    print("push report:", report_live)

    got_nodes = mc.get_nodes()
    got_elems = mc.get_elements()
    print(f"read back: {len(got_nodes)} nodes, {len(got_elems)} elements")
    assert len(got_nodes) == len(model.nodes), "node count mismatch on MIDAS side"
    assert len(got_elems) == len(model.elements), "element count mismatch"
    print("PASS — round-trip counts match the hub")
else:
    print("Dry run (RUN_MIDAS=False). Would send:")
    for t, body in payloads.items():
        print(f"  PUT /db/{t:5s}  {len(body)} item(s)")
    print("  + real SECT/MATL from hub_section_material_blocks")

Dry run (RUN_MIDAS=False). Would send:
  PUT /db/UNIT   1 item(s)
  PUT /db/MATL   1 item(s)
  PUT /db/SECT   1 item(s)
  PUT /db/NODE   55 item(s)
  PUT /db/ELEM   50 item(s)
  PUT /db/CONS   25 item(s)
  PUT /db/STLD   1 item(s)
  PUT /db/CNLD   55 item(s)
  + real SECT/MATL from hub_section_material_blocks


## 6 · Summary

| Object | Rhino source (`gdr.*`) | MIDAS table | Verified |
|---|---|---|---|
| Nodes | curve vertices + split points | `NODE` | count, ids, coordinates |
| Elements | `gdr.kind=girder` segments | `ELEM` | count, node refs, BEAM type |
| Sections | `gdr.shape` (AISC label) | `SECT` | one per shape, real AISC dims |
| Materials | `gdr.grade` | `MATL` | one per grade |
| Supports | `gdr.kind=support` + `gdr.fixity` | `CONS` | count, 7-flag string, fixed vs expansion |
| Loads | deck/parapet DC (from `build_deck`) | `STLD` + `CNLD` | case present, nodal loads sent |

The splice markers (`gdr.kind=splice`) and the deck/parapet **geometry** do not
become MIDAS elements — splices are a design/review artifact and the deck acts
through the composite section and the DC loads shown above. Everything that
*is* structure in the analysis model transfers and checks out.